In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score
from sklearn.neighbors import NearestNeighbors


from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
pd.set_option('display.max_columns', None)

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the East-West Airlines dataset (the 'data' sheet contains the records)
file_path = 'EastWestAirlines.xlsx'  # update path if needed
df = pd.read_excel(file_path, sheet_name='data')

print('Shape of dataset:', df.shape)
df.head()

In [ ]:
# Clean up column names (remove special characters, extra spaces)
df.columns = df.columns.str.strip().str.replace('#', '', regex=False).str.replace('?', '', regex=False)
print(df.columns.tolist())

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
# Check for missing values
missing = df.isnull().sum()
print('Missing values per column:')
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values found.')

In [ ]:
# If there were missing values, we would impute them (median for numeric features is robust to skew/outliers)
# This is included defensively in case the dataset version you have contains nulls.
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(df[col].median())

print('Missing values after imputation:', df.isnull().sum().sum())

In [ ]:
# Keep ID aside, drop it from the feature set used for clustering
id_col = 'ID' if 'ID' in df.columns else df.columns[0]
customer_ids = df[id_col].copy()

features_df = df.drop(columns=[id_col])
print('Features used for clustering:', features_df.columns.tolist())

In [ ]:
# Boxplots to visualize outliers across all numeric features
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(features_df.columns):
    sns.boxplot(y=features_df[col], ax=axes[i], color='skyblue')
    axes[i].set_title(col)
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.suptitle('Boxplots of Raw Features (Outlier Inspection)', y=1.02, fontsize=14)
plt.show()

In [ ]:
# Cap outliers using the IQR method (1.5x IQR rule) - applied to continuous, skewed columns
def cap_outliers_iqr(series, k=1.5):
    Q1, Q3 = series.quantile(0.25), series.quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - k*IQR, Q3 + k*IQR
    return series.clip(lower=lower, upper=upper)

# Columns with continuous, heavily skewed distributions (the count-like cc1/cc2/cc3_miles and Award are left untouched)
skewed_cols = ['Balance', 'Qual_miles', 'Bonus_miles', 'Bonus_trans',
               'Flight_miles_12mo', 'Flight_trans_12']
skewed_cols = [c for c in skewed_cols if c in features_df.columns]

features_capped = features_df.copy()
for col in skewed_cols:
    features_capped[col] = cap_outliers_iqr(features_capped[col])

print('Outliers capped for:', skewed_cols)
features_capped[skewed_cols].describe().T

In [ ]:
# Boxplots after capping
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
for i, col in enumerate(skewed_cols):
    sns.boxplot(y=features_capped[col], ax=axes[i], color='lightgreen')
    axes[i].set_title(col)
plt.tight_layout()
plt.suptitle('Boxplots After IQR Capping', y=1.02, fontsize=14)
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(features_capped)
X_scaled_df = pd.DataFrame(X_scaled, columns=features_capped.columns)

X_scaled_df.describe().T[['mean', 'std', 'min', 'max']]

In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()
for i, col in enumerate(features_capped.columns):
    sns.histplot(features_capped[col], kde=True, ax=axes[i], color='teal')
    axes[i].set_title(f'Distribution of {col}')
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 9))
corr = features_capped.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True, linewidths=0.5)
plt.title('Correlation Heatmap of Features')
plt.tight_layout()
plt.show()

In [ ]:
key_vars = ['Balance', 'Bonus_miles', 'Flight_miles_12mo', 'Days_since_enroll']
key_vars = [c for c in key_vars if c in features_capped.columns]
sns.pairplot(features_capped[key_vars], diag_kind='kde', plot_kws={'alpha': 0.4, 's': 15})
plt.suptitle('Pairwise Relationships of Key Variables', y=1.02)
plt.show()

In [ ]:
pca_explore = PCA(n_components=2, random_state=42)
pca_coords_explore = pca_explore.fit_transform(X_scaled)
print('Explained variance ratio (PC1, PC2):', pca_explore.explained_variance_ratio_)
print('Cumulative variance explained:', pca_explore.explained_variance_ratio_.sum().round(3))

plt.figure(figsize=(8, 6))
plt.scatter(pca_coords_explore[:, 0], pca_coords_explore[:, 1], alpha=0.4, s=15, color='steelblue')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('PCA Projection of Scaled Data (Pre-Clustering)')
plt.show()

In [ ]:
inertia = []
sil_scores_kmeans = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=42)
    labels_k = km.fit_predict(X_scaled)
    inertia.append(km.inertia_)
    sil_scores_kmeans.append(silhouette_score(X_scaled, labels_k))

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(list(K_range), inertia, marker='o', color='darkorange')
axes[0].set_xlabel('Number of Clusters (K)')
axes[0].set_ylabel('Inertia (WCSS)')
axes[0].set_title('Elbow Method for Optimal K')

axes[1].plot(list(K_range), sil_scores_kmeans, marker='o', color='seagreen')
axes[1].set_xlabel('Number of Clusters (K)')
axes[1].set_ylabel('Silhouette Score')
axes[1].set_title('Silhouette Score vs K')

plt.tight_layout()
plt.show()

best_k = list(K_range)[int(np.argmax(sil_scores_kmeans))]
print(f'K with highest silhouette score: {best_k}')

In [ ]:
final_k = best_k  # override manually here if elbow plot suggests a different K, e.g. final_k = 4

kmeans_model = KMeans(n_clusters=final_k, init='k-means++', n_init=10, random_state=42)
kmeans_labels = kmeans_model.fit_predict(X_scaled)

df['KMeans_Cluster'] = kmeans_labels

print(f'Final K = {final_k}')
print('Cluster sizes:')
print(pd.Series(kmeans_labels).value_counts().sort_index())
print(f'\nSilhouette Score: {silhouette_score(X_scaled, kmeans_labels):.4f}')
print(f'Calinski-Harabasz Score: {calinski_harabasz_score(X_scaled, kmeans_labels):.2f}')

In [ ]:
sil_vals = silhouette_samples(X_scaled, kmeans_labels)

fig, ax = plt.subplots(figsize=(9, 6))
y_lower = 10
colors = sns.color_palette('tab10', final_k)
for i in range(final_k):
    cluster_sil_vals = np.sort(sil_vals[kmeans_labels == i])
    size_i = cluster_sil_vals.shape[0]
    y_upper = y_lower + size_i
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_sil_vals, facecolor=colors[i], alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size_i, str(i))
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(X_scaled, kmeans_labels), color='red', linestyle='--', label='Average Score')
ax.set_xlabel('Silhouette Coefficient')
ax.set_ylabel('Cluster')
ax.set_title('Silhouette Plot — K-Means')
ax.legend()
plt.show()

In [ ]:
# Subsample for a readable dendrogram (visualization only)
np.random.seed(42)
sample_size = min(200, X_scaled.shape[0])
sample_idx = np.random.choice(X_scaled.shape[0], sample_size, replace=False)
X_sample = X_scaled[sample_idx]

linked = linkage(X_sample, method='ward')

plt.figure(figsize=(14, 6))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90., leaf_font_size=9, show_contracted=True)
plt.title(f'Hierarchical Clustering Dendrogram (Ward Linkage, n={sample_size} sample)')
plt.xlabel('Sample Index / Cluster Size')
plt.ylabel('Distance')
plt.show()

In [ ]:
agg_model = AgglomerativeClustering(n_clusters=final_k, linkage='ward')
agg_labels = agg_model.fit_predict(X_scaled)

df['Hierarchical_Cluster'] = agg_labels

print('Cluster sizes:')
print(pd.Series(agg_labels).value_counts().sort_index())
print(f'\nSilhouette Score: {silhouette_score(X_scaled, agg_labels):.4f}')
print(f'Calinski-Harabasz Score: {calinski_harabasz_score(X_scaled, agg_labels):.2f}')

In [ ]:
n_features = X_scaled.shape[1]
min_pts = 2 * n_features
print(f'Number of features: {n_features} -> using min_samples (minPts) = {min_pts}')

neighbors = NearestNeighbors(n_neighbors=min_pts)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(9, 6))
plt.plot(k_distances, color='purple')
plt.xlabel('Points sorted by distance')
plt.ylabel(f'{min_pts}-th Nearest Neighbor Distance')
plt.title('K-Distance Plot for DBSCAN eps Selection')
plt.grid(True)
plt.show()

print('Inspect the plot above for the "elbow" / knee point — that y-value is a good eps candidate.')

In [ ]:
# TIP: adjust eps based on the elbow you observe in the k-distance plot above.
eps_value = 1.5      # <-- tune this based on the k-distance elbow
min_samples_value = min_pts

dbscan_model = DBSCAN(eps=eps_value, min_samples=min_samples_value)
dbscan_labels = dbscan_model.fit_predict(X_scaled)

df['DBSCAN_Cluster'] = dbscan_labels

n_clusters_db = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise = list(dbscan_labels).count(-1)

print(f'eps = {eps_value}, min_samples = {min_samples_value}')
print(f'Number of clusters found (excluding noise): {n_clusters_db}')
print(f'Number of noise points: {n_noise} ({100*n_noise/len(dbscan_labels):.1f}% of data)')
print('\nCluster sizes (including -1 = noise):')
print(pd.Series(dbscan_labels).value_counts().sort_index())

In [ ]:
# Silhouette score for DBSCAN is only meaningful if there are >=2 clusters AND not all points are noise
mask = dbscan_labels != -1
if n_clusters_db >= 2 and mask.sum() > 0:
    sil_dbscan = silhouette_score(X_scaled[mask], dbscan_labels[mask])
    print(f'Silhouette Score (excluding noise points): {sil_dbscan:.4f}')
else:
    print('Not enough clusters (excluding noise) to compute a silhouette score.')
    print('Try decreasing eps_value or re-checking the k-distance elbow.')

In [ ]:
eps_grid = np.arange(0.5, 3.1, 0.25)
results = []

for eps_try in eps_grid:
    labels_try = DBSCAN(eps=eps_try, min_samples=min_pts).fit_predict(X_scaled)
    n_clust = len(set(labels_try)) - (1 if -1 in labels_try else 0)
    noise_pct = 100 * list(labels_try).count(-1) / len(labels_try)
    if n_clust >= 2:
        mask_try = labels_try != -1
        sil = silhouette_score(X_scaled[mask_try], labels_try[mask_try])
    else:
        sil = np.nan
    results.append({'eps': round(eps_try, 2), 'n_clusters': n_clust, 'noise_pct': round(noise_pct, 1), 'silhouette': sil})

results_df = pd.DataFrame(results)
results_df

In [ ]:
pca = PCA(n_components=2, random_state=42)
pca_coords = pca.fit_transform(X_scaled)

fig, axes = plt.subplots(1, 3, figsize=(20, 6))

sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=kmeans_labels, palette='tab10', s=20, alpha=0.7, ax=axes[0], legend='full')
axes[0].set_title(f'K-Means (K={final_k})')
axes[0].set_xlabel('PC1'); axes[0].set_ylabel('PC2')

sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=agg_labels, palette='tab10', s=20, alpha=0.7, ax=axes[1], legend='full')
axes[1].set_title(f'Hierarchical (Agglomerative, K={final_k})')
axes[1].set_xlabel('PC1'); axes[1].set_ylabel('PC2')

sns.scatterplot(x=pca_coords[:, 0], y=pca_coords[:, 1], hue=dbscan_labels, palette='tab10', s=20, alpha=0.7, ax=axes[2], legend='full')
axes[2].set_title(f'DBSCAN (eps={eps_value}, minPts={min_samples_value})')
axes[2].set_xlabel('PC1'); axes[2].set_ylabel('PC2')

plt.tight_layout()
plt.show()

In [ ]:
comparison = []

comparison.append({
    'Algorithm': 'K-Means',
    'Num_Clusters': final_k,
    'Silhouette_Score': silhouette_score(X_scaled, kmeans_labels),
    'Calinski_Harabasz': calinski_harabasz_score(X_scaled, kmeans_labels),
    'Noise_Points': 0
})

comparison.append({
    'Algorithm': 'Hierarchical',
    'Num_Clusters': final_k,
    'Silhouette_Score': silhouette_score(X_scaled, agg_labels),
    'Calinski_Harabasz': calinski_harabasz_score(X_scaled, agg_labels),
    'Noise_Points': 0
})

if n_clusters_db >= 2:
    mask = dbscan_labels != -1
    comparison.append({
        'Algorithm': 'DBSCAN',
        'Num_Clusters': n_clusters_db,
        'Silhouette_Score': silhouette_score(X_scaled[mask], dbscan_labels[mask]),
        'Calinski_Harabasz': calinski_harabasz_score(X_scaled[mask], dbscan_labels[mask]),
        'Noise_Points': int((dbscan_labels == -1).sum())
    })

comparison_df = pd.DataFrame(comparison)
comparison_df

In [ ]:
plt.figure(figsize=(8, 5))
sns.barplot(data=comparison_df, x='Algorithm', y='Silhouette_Score', palette='viridis')
plt.title('Silhouette Score Comparison Across Algorithms')
plt.ylabel('Silhouette Score')
plt.axhline(0, color='gray', linewidth=0.8)
plt.show()

In [ ]:
cluster_profile = df.groupby('KMeans_Cluster')[features_df.columns].mean().round(1)
cluster_profile['Cluster_Size'] = df['KMeans_Cluster'].value_counts().sort_index()
cluster_profile

In [ ]:
# Heatmap of standardized cluster centers - makes it easy to see which features define each cluster
cluster_centers_scaled = pd.DataFrame(
    scaler.transform(cluster_profile[features_df.columns]),
    columns=features_df.columns,
    index=cluster_profile.index
)

plt.figure(figsize=(11, 5))
sns.heatmap(cluster_centers_scaled, annot=True, fmt='.2f', cmap='RdBu_r', center=0)
plt.title('K-Means Cluster Profiles (Standardized Feature Means)')
plt.ylabel('Cluster')
plt.show()

In [ ]:
output_path = 'EastWestAirlines_with_clusters.csv'
df.to_csv(output_path, index=False)
print(f'Saved results to {output_path}')
df[[id_col, 'KMeans_Cluster', 'Hierarchical_Cluster', 'DBSCAN_Cluster']].head(10)